# Clínica - Projeto Itaú

In [27]:
import json
import os
import re
import unicodedata
import random
import pandas as pd
import asyncio
import hashlib
import time
from pathlib import Path

import nest_asyncio

Entrada e normalização
Leitura do CSV e ajuste inicial de texto/codificação.

In [28]:
# LINHA DE SELEÇÃO DO INPUT
df = pd.read_csv("dataset_clinica_20261.csv", encoding="utf-8") # 4s para carregar

def corrigir_mojibake(valor):
    if isinstance(valor, str) and ("Ã" in valor or "Â" in valor):
        try:
            return valor.encode("latin1").decode("utf-8")
        except UnicodeError:
            return valor
    return valor

colunas_texto = df.select_dtypes(include="object").columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(corrigir_mojibake))

if "magistrado" in df.columns:
    df["magistrado"] = df["magistrado"].astype(str).str.strip().str.upper()

display(df.head())
df.info()

C:\Users\gabriel\AppData\Local\Temp\ipykernel_3396\2772562578.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns


,cd_processo,id_processo,classe,assunto,magistrado,comarca,foro,vara,data_disponibilizacao,decisao
0,5B0005D030000,1002017-64.2024.8.26.0191,Procedimento do Juizado Especial Cível,Bancários,LUCIANA DO CARMO NOGUEIRA,Ferraz de Vasconcelos,Foro de Ferraz de Vasconcelos,Vara do Juizado Especial Cível e Criminal,27/08/2024,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...
1,03001EPSV0000,1034196-67.2023.8.26.0003,Procedimento Comum Cível,Bancários,LAURA MOTA LIMA DE OLIVEIRA BACCIN,SÃO PAULO,Foro Regional III - Jabaquara,1ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...
2,I20002VT10000,1000247-51.2023.8.26.0650,Procedimento Comum Cível,Empréstimo consignado,MARCIA YOSHIE ISHIKAWA,Valinhos,Foro de Valinhos,3ª Vara,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...
3,DE000DZ0N0000,1010110-16.2024.8.26.0482,Procedimento Comum Cível,Bancários,LEONARDO MAZZILLI MARCONDES,Presidente Prudente,Foro de Presidente Prudente,4ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...
4,2S001VMCT0000,1101723-02.2024.8.26.0100,Procedimento Comum Cível,Bancários,MICHELLE FABIOLA DITTERT PUPULIM,SÃO PAULO,Foro Regional III - Jabaquara,6ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...


<class 'pandas.DataFrame'>
RangeIndex: 22866 entries, 0 to 22865
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   cd_processo            22866 non-null  str  
 1   id_processo            22866 non-null  str  
 2   classe                 22856 non-null  str  
 3   assunto                22866 non-null  str  
 4   magistrado             22866 non-null  str  
 5   comarca                22866 non-null  str  
 6   foro                   22866 non-null  str  
 7   vara                   22866 non-null  str  
 8   data_disponibilizacao  22866 non-null  str  
 9   decisao                22866 non-null  str  
dtypes: str(10)
memory usage: 181.1 MB


Preparação da amostra para análise com IA

100 casos aleatórios (reproduzíveis via `random_state=42`).

In [29]:
df_curto = df.copy()
#df_curto = df_curto.sample(n=1000, random_state=42).reset_index(drop=True)

In [30]:
df_curto.shape

(22866, 10)

Variáveis para extração

- justiça_gratuita (sim/não)
- rito_processual (Juizado Especial / Procedimento Comum)
- tipo_acao (fraude/golpe, cobrança indevida, empréstimo não reconhecido, revisão contratual)
- contato_previo_banco (sim/não)
- canal_contato (SAC, Ouvidoria, Procon, Reclame Aqui, Agência, não identificado)
- mencao_reclame_aqui (sim/não)
- boletim_de_ocorrencia (sim/não)
- resultado_julgamento (procedente, improcedente, parcialmente procedente, extinto)
- culpa_atribuida (banco, consumidor, terceiro, compartilhada)
- valor_danos_morais (float, R$)
- valor_danos_materiais (float, R$)


## Extração com OpenAI — async
Extrai as variáveis em JSON com cache e retry.

In [31]:
"""
COMENTADO: Extração com OpenAI — async
Esta célula estava fazendo chamadas à API OpenAI. 
Agora carregamos apenas do cache (ver célula seguinte).

nest_asyncio.apply()

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Defina OPENAI_API_KEY antes de executar esta célula.")

async_client = AsyncOpenAI(api_key=api_key)
modelo_openai = "gpt-4.1-mini"
MAX_CONCORRENTE = 20
SALVAR_CACHE_A_CADA = 50

# --- Cache local ---
CACHE_PATH = Path("cache") / "cache_openai.json"
CACHE_PATH.parent.mkdir(exist_ok=True)
"""

'\nCOMENTADO: Extração com OpenAI — async\nEsta célula estava fazendo chamadas à API OpenAI. \nAgora carregamos apenas do cache (ver célula seguinte).\n\nnest_asyncio.apply()\n\ntry:\n    from tqdm.notebook import tqdm\nexcept ImportError:\n    from tqdm import tqdm\n\nfrom openai import AsyncOpenAI\nfrom dotenv import load_dotenv\n\nload_dotenv()\n\napi_key = os.getenv("OPENAI_API_KEY")\nif not api_key:\n    raise ValueError("Defina OPENAI_API_KEY antes de executar esta célula.")\n\nasync_client = AsyncOpenAI(api_key=api_key)\nmodelo_openai = "gpt-4.1-mini"\nMAX_CONCORRENTE = 20\nSALVAR_CACHE_A_CADA = 50\n\n# --- Cache local ---\nCACHE_PATH = Path("cache") / "cache_openai.json"\nCACHE_PATH.parent.mkdir(exist_ok=True)\n'

In [32]:
"""
COMENTADO: Funções de cache, prompts e chamadas assíncronas à OpenAI
Toda esta lógica foi comentada. Agora carregamos apenas do cache.

def _carregar_cache():
    if CACHE_PATH.exists():
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def _salvar_cache(cache):
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

cache_local = _carregar_cache()
print(f"Cache carregado: {len(cache_local)} entradas existentes")

# --- Prompts ---
prompt_sistema = (
    "Você é um analista jurídico especializado em processos cíveis contra bancos. "
    "Extraia informações estruturadas de decisões judiciais em português. "
    "Responda APENAS com JSON válido, sem texto adicional."
)

template_prompt = ...
_campos_padrao = {...}

def _hash_decisao(texto):
    ...

def _parse_json_seguro(conteudo):
    ...

async def _chamar_decisao_async(idx, texto, semaforo, max_tentativas=4):
    ...

async def processar_async(indices_api, decisoes, resultados_final):
    ...

# --- Separa cached dos que precisam de API ---
# [toda a lógica de processamento assíncrono estava aqui]
"""

'\nCOMENTADO: Funções de cache, prompts e chamadas assíncronas à OpenAI\nToda esta lógica foi comentada. Agora carregamos apenas do cache.\n\ndef _carregar_cache():\n    if CACHE_PATH.exists():\n        with open(CACHE_PATH, "r", encoding="utf-8") as f:\n            return json.load(f)\n    return {}\n\ndef _salvar_cache(cache):\n    with open(CACHE_PATH, "w", encoding="utf-8") as f:\n        json.dump(cache, f, ensure_ascii=False, indent=2)\n\ncache_local = _carregar_cache()\nprint(f"Cache carregado: {len(cache_local)} entradas existentes")\n\n# --- Prompts ---\nprompt_sistema = (\n    "Você é um analista jurídico especializado em processos cíveis contra bancos. "\n    "Extraia informações estruturadas de decisões judiciais em português. "\n    "Responda APENAS com JSON válido, sem texto adicional."\n)\n\ntemplate_prompt = ...\n_campos_padrao = {...}\n\ndef _hash_decisao(texto):\n    ...\n\ndef _parse_json_seguro(conteudo):\n    ...\n\nasync def _chamar_decisao_async(idx, texto, semaf

Reprocessamento dos erros

In [33]:
"""
COMENTADO: Reprocessamento dos erros
Esta célula reprocessava entradas com ERRO. Mantida comentada para testes futuros.

## Reprocessamento

# Identifica linhas com ERRO no resultado
mask_erros = df_curto['resultado_julgamento_ia'].astype(str).str.startswith('ERRO', na=False)
indices_erro = df_curto[mask_erros].index.tolist()
print(f"Linhas com ERRO para reprocessar: {len(indices_erro)}")

if indices_erro:
    # Pega os textos e limpa o cache dessas entradas (força nova chamada)
    decisoes_erro = df_curto.loc[indices_erro, 'decisao'].fillna('').tolist()
    for texto in decisoes_erro:
        hash_key = _hash_decisao(texto)
        if hash_key in cache_local:
            del cache_local[hash_key]

    # Reprocessa só os erros
    resultados_reprocess = [None] * len(decisoes_erro)
    indices_todos = list(range(len(decisoes_erro)))

    t_inicio = time.time()
    await processar_async(indices_todos, decisoes_erro, resultados_reprocess)
    tempo_total = time.time() - t_inicio
    print(f"Reprocessamento concluído em {tempo_total:.1f}s")

    # Atualiza df_curto com os novos resultados
    df_novos = pd.DataFrame(resultados_reprocess).rename(
        columns={c: f"{c}_ia" for c in _campos_padrao.keys()}
    )

    colunas_ia = [f"{c}_ia" for c in _campos_padrao.keys()]
    df_curto.loc[indices_erro, colunas_ia] = df_novos[colunas_ia].values

    # Verifica se ainda restam erros
    erros_restantes = df_curto['resultado_julgamento_ia'].astype(str).str.startswith('ERRO', na=False).sum()
    print(f"Erros restantes após reprocessamento: {erros_restantes}")
    pasta_saida = "output"
    os.makedirs(pasta_saida, exist_ok=True)
    # Salva o arquivo atualizado
    arquivo_saida = os.path.join(pasta_saida, "curto_ia.xlsx")
    df_curto.to_excel(arquivo_saida, index=False, sheet_name="Processos")
    print("Arquivo atualizado salvo: output/curto_ia.xlsx")
else:
    pasta_saida = "output"
    os.makedirs(pasta_saida, exist_ok=True)
    # Salva o arquivo atualizado
    arquivo_saida = os.path.join(pasta_saida, "curto_ia.xlsx")
    df_curto.to_excel(arquivo_saida, index=False, sheet_name="Processos")
    print("Arquivo salvo: output/curto_ia.xlsx")
    print("Nenhum erro encontrado — nada a reprocessar.")
"""

'\nCOMENTADO: Reprocessamento dos erros\nEsta célula reprocessava entradas com ERRO. Mantida comentada para testes futuros.\n\n## Reprocessamento\n\n# Identifica linhas com ERRO no resultado\nmask_erros = df_curto[\'resultado_julgamento_ia\'].astype(str).str.startswith(\'ERRO\', na=False)\nindices_erro = df_curto[mask_erros].index.tolist()\nprint(f"Linhas com ERRO para reprocessar: {len(indices_erro)}")\n\nif indices_erro:\n    # Pega os textos e limpa o cache dessas entradas (força nova chamada)\n    decisoes_erro = df_curto.loc[indices_erro, \'decisao\'].fillna(\'\').tolist()\n    for texto in decisoes_erro:\n        hash_key = _hash_decisao(texto)\n        if hash_key in cache_local:\n            del cache_local[hash_key]\n\n    # Reprocessa só os erros\n    resultados_reprocess = [None] * len(decisoes_erro)\n    indices_todos = list(range(len(decisoes_erro)))\n\n    t_inicio = time.time()\n    await processar_async(indices_todos, decisoes_erro, resultados_reprocess)\n    tempo_tota

## Classificação por Regex — baseline de comparação

Para cada variável extraída pela IA, criamos um classificador por regex aplicado ao texto bruto da decisão.
Objetivo: medir o **acordo IA vs regex**. Alta concordância → regex é suficiente. Baixa concordância → IA agrega valor real.

In [ ]:
# ── Aplica tudo com MULTIPROCESSING (5-8x mais rápido) ──────────────────────────
from multiprocessing import Pool
import time as time_module

def _aplicar_paralelo(func, series_list, n_workers=8):
    """Aplica função em paralelo usando todos os cores"""
    with Pool(n_workers) as pool:
        resultado = pool.map(func, series_list)
    return resultado

decisao_list = df_curto["decisao"].fillna("").tolist()

print("Processando regex em paralelo (8 cores)...")
t_inicio = time_module.time()

# Define pares (coluna_nome, função_regex)
regex_funcs = [
    ("tipo_documento_regex", regex_tipo_documento),
    ("fora_do_escopo_regex", regex_fora_do_escopo),
    ("justica_gratuita_regex", regex_justica_gratuita),
    ("rito_processual_regex", regex_rito_processual),
    ("tipo_acao_regex", regex_tipo_acao),
    ("contato_previo_banco_regex", regex_contato_previo),
    ("canal_contato_regex", regex_canal_contato),
    ("mencao_reclame_aqui_regex", regex_mencao_reclame),
    ("boletim_de_ocorrencia_regex", regex_boletim),
    ("resultado_julgamento_regex", regex_resultado),
    ("culpa_atribuida_regex", regex_culpa),
    ("valor_danos_morais_regex", regex_valor_morais),
    ("valor_danos_materiais_regex", regex_valor_materiais),
    ("repeticao_indebito_regex", regex_repeticao_indebito),
]

# Aplica em paralelo
for col_nome, func in regex_funcs:
    df_curto[col_nome] = _aplicar_paralelo(func, decisao_list, n_workers=8)
    print(f"  ✓ {col_nome}")

tempo_total = time_module.time() - t_inicio
print(f"\n✓ Concluído em {tempo_total:.1f}s")
print(f"Colunas _regex adicionadas: {len([c for c in df_curto.columns if c.endswith('_regex')])}")

Processando regex em paralelo (8 cores)...


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CARREGAR APENAS DO CACHE (novo fluxo)
# ═══════════════════════════════════════════════════════════════════════════

CACHE_PATH = Path("cache") / "cache_openai.json"

def _carregar_cache():
    if CACHE_PATH.exists():
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

_campos_padrao = {
    "tipo_documento": "não identificado",
    "fora_do_escopo": False,
    "justica_gratuita": "não",
    "rito_processual": "não identificado",
    "tipo_acao": "não identificado",
    "contato_previo_banco": "não",
    "canal_contato": "não identificado",
    "mencao_reclame_aqui": "não",
    "boletim_de_ocorrencia": "não",
    "resultado_julgamento": "não identificado",
    "culpa_atribuida": "não identificado",
    "valor_danos_morais": 0.0,
    "valor_danos_materiais": 0.0,
    "repetição_indébito": "não identificado",
}

def _hash_decisao(texto):
    chave = texto[:500].strip()
    return hashlib.md5(chave.encode("utf-8")).hexdigest()

cache_local = _carregar_cache()
print(f"✓ Cache carregado: {len(cache_local)} entradas")

# Mapeia cada decisão para o seu resultado em cache
decisoes = df_curto["decisao"].fillna("").tolist()
resultados_final = []

hits = 0
for i, texto in enumerate(decisoes):
    hash_key = _hash_decisao(texto)
    resultado = cache_local.get(hash_key)
    
    if resultado is not None:
        resultados_final.append({**_campos_padrao, **resultado})
        hits += 1
    else:
        resultados_final.append({**_campos_padrao})

print(f"✓ Cache hits: {hits}/{len(decisoes)} ({hits/len(decisoes)*100:.1f}%)")

# Concatena com df_curto
df_ai = pd.DataFrame(resultados_final).rename(
    columns={c: f"{c}_ia" for c in _campos_padrao.keys()}
)

colunas_originais = [c for c in df_curto.columns if not c.endswith("_ia")]
df_curto = df_curto[colunas_originais]
df_curto = pd.concat([df_curto.reset_index(drop=True), df_ai.reset_index(drop=True)], axis=1)

print(f"✓ DataFrame pronto: {len(df_curto)} linhas × {len(df_curto.columns)} colunas")
print(f"  Colunas _ia: {len([c for c in df_curto.columns if c.endswith('_ia')])}")
df_curto.head(3)

Comparacao:

In [ ]:
## Comparação IA vs Regex
# Campos comparáveis (exclui valores numéricos — têm métrica própria)
campos_categoricos = [
    "tipo_documento",
    "justica_gratuita",
    "rito_processual",
    "tipo_acao",
    "contato_previo_banco",
    "canal_contato",
    "mencao_reclame_aqui",
    "boletim_de_ocorrencia",
    "resultado_julgamento",
    "culpa_atribuida",
    "repetição_indébito",
]

# Exclui linhas com ERRO na IA e fora do escopo
mask_validos = (
    ~df_curto["resultado_julgamento_ia"].astype(str).str.startswith("ERRO") &
    (df_curto["fora_do_escopo_ia"] == False)
)
sub = df_curto[mask_validos].copy()

print(f"Linhas analisadas (sem erro, sem fora do escopo): {len(sub)}\n")
print(f"{'CAMPO':<30} {'IGUAIS':>7} {'DIFER.':>7} {'ACORDO %':>9}")
print("-" * 58)

rows = []
for campo in campos_categoricos:
    col_ia    = f"{campo}_ia"
    col_regex = f"{campo}_regex"
    if col_ia not in sub.columns or col_regex not in sub.columns:
        continue

    ia    = sub[col_ia].astype(str).str.strip().str.lower()
    regex = sub[col_regex].astype(str).str.strip().str.lower()

    iguais = (ia == regex).sum()
    total  = len(sub)
    acordo = iguais / total * 100

    print(f"{campo:<30} {iguais:>7} {total - iguais:>7} {acordo:>8.1f}%")
    rows.append({"campo": campo, "iguais": iguais, "diferentes": total - iguais, "acordo_%": round(acordo, 1)})

df_concordancia = pd.DataFrame(rows).sort_values("acordo_%", ascending=False)

# Comparação de valores numéricos
print("\n--- Valores numéricos (só linhas com condenação > 0 em ambos) ---")
for campo_val in ["valor_danos_morais", "valor_danos_materiais"]:
    col_ia    = f"{campo_val}_ia"
    col_regex = f"{campo_val}_regex"
    ambos_pos = sub[(sub[col_ia] > 0) & (sub[col_regex] > 0)]
    if len(ambos_pos) == 0:
        print(f"{campo_val}: sem casos com ambos > 0")
        continue
    diff_pct = ((ambos_pos[col_ia] - ambos_pos[col_regex]).abs() / ambos_pos[col_ia] * 100)
    print(f"{campo_val}: n={len(ambos_pos)}, diferença média={diff_pct.mean():.1f}%, mediana={diff_pct.median():.1f}%")

# Inspeção manual das divergências
print("\n--- Inspeção: campos com menor acordo ---")
campo_pior = df_concordancia.iloc[-1]["campo"]
col_ia    = f"{campo_pior}_ia"
col_regex = f"{campo_pior}_regex"
divergentes = sub[sub[col_ia].astype(str).str.lower() != sub[col_regex].astype(str).str.lower()]
print(f"\nCampo '{campo_pior}' — {len(divergentes)} divergências. Amostra de 5:\n")
display(
    divergentes[["id_processo", col_ia, col_regex]]
    .head(5)
    .rename(columns={col_ia: "IA", col_regex: "Regex"})
)

KeyError: 'resultado_julgamento_ia'

In [ ]:
# Exportar xlsx com colunas _ia e _regex intercaladas

colunas_base = [c for c in df_curto.columns if not c.endswith("_ia") and not c.endswith("_regex")]

campos_ia = [c for c in df_curto.columns if c.endswith("_ia")]
campos_regex = [c.replace("_ia", "_regex") for c in campos_ia if c.replace("_ia", "_regex") in df_curto.columns]

colunas_intercaladas = []
for col_ia in campos_ia:
    colunas_intercaladas.append(col_ia)
    col_regex = col_ia.replace("_ia", "_regex")
    if col_regex in df_curto.columns:
        colunas_intercaladas.append(col_regex)
   
ordem_final = colunas_base + colunas_intercaladas

pasta_saida = "output"
os.makedirs(pasta_saida, exist_ok=True)
arquivo_saida = os.path.join(pasta_saida, "curto_ia_regex.xlsx")
df_curto[ordem_final].to_excel(arquivo_saida, index=False, sheet_name="Processos")

print(f"Arquivo salvo: output/curto_ia_regex.xlsx")
print(f"Total colunas: {len(ordem_final)} ({len(colunas_base)} base + {len(colunas_intercaladas)} IA/regex)")
print("\nOrdem das colunas IA/regex:")
for i in range(0, len(colunas_intercaladas), 2):
    ia = colunas_intercaladas[i]
    rx = colunas_intercaladas[i+1] if i+1 < len(colunas_intercaladas) else "—"
    print(f"  {ia:<35} | {rx}")

In [ ]:
# Carrega o arquivo com IA e Regex para análise
df_analise = pd.read_excel("output/curto_ia_regex.xlsx", sheet_name="Processos")

In [ ]:
df_ana_t_doc = df_analise.copy()

# Divergências em tipo_documento
div = df_ana_t_doc[df_ana_t_doc["tipo_documento_ia"] != df_ana_t_doc["tipo_documento_regex"]].reset_index(drop=True)

print(f"Total divergências tipo_documento: {len(div)}")
display(div[["decisao","tipo_documento_ia", "tipo_documento_regex"]])

df_ana_t_doc=div[["decisao","tipo_documento_ia", "tipo_documento_regex"]]

arquivo_saida = os.path.join(pasta_saida, "df_ana_t_doc.xlsx")
df_ana_t_doc.to_excel(arquivo_saida, index=False, sheet_name="Processos")


In [ ]:
df_tip_a = df_analise.copy()

# Divergências em tipo_documento
div = df_tip_a[df_tip_a["tipo_acao_ia"] != df_tip_a["tipo_acao_regex"]].reset_index(drop=True)

display(div[["decisao","tipo_acao_ia", "tipo_acao_regex"]])

df_tip_a=div[["decisao","tipo_acao_ia", "tipo_acao_regex"]]

arquivo_saida = os.path.join(pasta_saida, "df_tip_a.xlsx")
df_tip_a.to_excel(arquivo_saida, index=False, sheet_name="Processos")
